# Credit Risk AI Workflow — Interactive Demo

This notebook demonstrates the full pipeline in 5 minutes:
1. Load a pre-trained model
2. Process a credit application
3. See the decision, SHAP factors, memo, and safety checks

**No API keys required** — LLM calls are mocked for this demo.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
os.environ.setdefault('OPENAI_API_KEY', 'demo-not-needed')

from unittest.mock import patch, MagicMock
import pandas as pd

## 1. Load the Pre-Trained Model

We ship XGBoost models trained on real data. Let's use the UCI model (Taiwan credit card defaults, 30K rows, AUC 0.769).

In [ ]:
import joblib

model = joblib.load('../models/uci_xgboost_v1.joblib')
print(f'Model type: {type(model).__name__}')
print(f'Pipeline steps: {[name for name, _ in model.steps]}')

## 2. Create a Sample Applicant

This is a borrower with late payments (PAY_0=2 means 2 months late) and moderate credit limit.

In [ ]:
applicant = {
    'LIMIT_BAL': 50000,
    'SEX': 2,           # 1=male, 2=female
    'EDUCATION': 2,     # 1=grad school, 2=university, 3=high school
    'MARRIAGE': 1,      # 1=married, 2=single
    'AGE': 35,
    'PAY_0': 2,         # 2 months late on most recent payment!
    'PAY_2': 0, 'PAY_3': 0, 'PAY_4': 0, 'PAY_5': 0, 'PAY_6': 0,
    'BILL_AMT1': 12000, 'BILL_AMT2': 11500, 'BILL_AMT3': 11000,
    'BILL_AMT4': 10500, 'BILL_AMT5': 10000, 'BILL_AMT6': 9500,
    'PAY_AMT1': 500, 'PAY_AMT2': 500, 'PAY_AMT3': 500,
    'PAY_AMT4': 500, 'PAY_AMT5': 500, 'PAY_AMT6': 500,
}

# Quick prediction
df = pd.DataFrame([applicant])
prob = model.predict_proba(df)[:, 1][0]
print(f'Default probability: {prob:.3f}')
print(f'Decision: {"DENY" if prob >= 0.5 else "APPROVE"}')

## 3. Run the Full Workflow

The `CreditWorkflow` orchestrates: predict → SHAP → memo → adverse action → safety checks.

We mock the LLM so this works without API keys.

In [ ]:
from workflow import CreditWorkflow, US

# Mock the LLM — in production, this calls GPT-4o or Claude
MOCK_MEMO = """CREDIT MEMO

Executive Summary: This applicant presents elevated default risk (probability: 0.72)
driven primarily by recent payment delinquency.

Risk Assessment:
- PAY_0 (most recent payment status): 2 months delinquent — strongest risk signal
- LIMIT_BAL ($50,000): Moderate credit limit relative to billing amounts
- PAY_AMT1 ($500): Low payment amounts relative to outstanding balance of $12,000

Recommendation: DENY. Primary driver is payment history delinquency."""

MOCK_DENIAL = """We were unable to approve your credit application. The primary reasons are:

1. Recent payment history shows delinquency (2 months past due)
2. Payment amounts are low relative to outstanding balances
3. Credit utilization pattern suggests repayment difficulty"""

# Build workflow with mocked LLM
with patch('workflow.pipeline.joblib.load', return_value=model):
    with patch('workflow.pipeline.OpenAI') as mock_openai:
        mock_openai.return_value = MagicMock()
        wf = CreditWorkflow(
            model_path='models/uci_xgboost_v1.joblib',
            llm_provider='openai',
            llm_model='gpt-4o',
            jurisdiction=US(),
        )

# Mock the LLM calls to return our pre-written text
call_count = [0]
def mock_llm(prompt):
    call_count[0] += 1
    return MOCK_MEMO if call_count[0] == 1 else MOCK_DENIAL

with patch.object(wf, '_call_llm', side_effect=mock_llm):
    result = wf.process_application(applicant)

print(f'Decision: {result.decision}')
print(f'Default probability: {result.probability:.3f}')
print(f'Flags: {result.flags}')
print(f'Jurisdiction: {result.metadata["jurisdiction"]}')

## 4. SHAP Factors — Why the Model Decided This

The model's top risk drivers, computed via SHAP (SHapley Additive exPlanations):

In [ ]:
print('Top SHAP factors (model explanation):\n')
for i, factor in enumerate(result.shap_factors[:5], 1):
    if 'error' not in factor:
        print(f"  {i}. {factor['feature']:20s} SHAP={factor['shap_value']:+.4f}  ({factor['direction']})")
    else:
        print(f"  {i}. {factor['feature']}: {factor.get('error', '')}")

## 5. Credit Memo (LLM-Generated, SHAP-Grounded)

The LLM writes the memo but can **only cite factors SHAP identified**:

In [ ]:
print(result.memo)

## 6. Adverse Action Notice (FCRA-Compliant)

For denials, the system generates a consumer-facing notice. The legal disclosure is **injected by code** (never written by the LLM):

In [ ]:
if result.adverse_action:
    # Show where the LLM text ends and the deterministic disclosure begins
    parts = result.adverse_action.split('\n\n---\n')
    print('=== LLM-GENERATED BODY ===\n')
    print(parts[0])
    print('\n=== DETERMINISTIC FCRA DISCLOSURE (injected by code) ===\n')
    if len(parts) > 1:
        print('---\n' + parts[1])
else:
    print('No adverse action (application approved)')

## 7. Safety Checks

Five layers protect the output before it reaches the consumer:

In [ ]:
print('Safety layer results:')
print(f'  Protected attribute hits: {result.metadata["protected_attribute_hits"]}')
print(f'  Flags triggered: {result.flags}')
print(f'  Escalated to human: {result.metadata["escalated"]}')
print(f'  Processing time: {result.metadata["processing_time_ms"]:.0f} ms')
print(f'  Decision ID: {result.metadata["decision_id"]}')

## 8. Switch Jurisdictions

The same pipeline works across 10 markets. Just swap the jurisdiction:

In [ ]:
from workflow.jurisdictions import ALL_JURISDICTIONS

print(f'Supported jurisdictions ({len(ALL_JURISDICTIONS)}):\n')
for code, cls in ALL_JURISDICTIONS.items():
    j = cls()
    print(f'  {code:4s} | {j.name:25s} | Regulator: {j.primary_regulator}')

## 9. Train Your Own Model

The SDK includes a configurable training pipeline:

```bash
# Download data + train in one command:
python -m workflow.training train --config configs/hmda_us.yaml

# Or bring your own data:
# See examples/bring_your_own_data/README.md
```

Pre-trained models available:
- `models/uci_xgboost_v1.joblib` — Taiwan credit cards (AUC 0.769)
- `models/hmda_xgboost_v1.joblib` — US mortgages (AUC 0.847)

In [ ]:
import json

# Show pre-trained model metrics
for model_file in ['uci_xgboost_v1_metrics.json', 'hmda_xgboost_v1_metrics.json']:
    path = f'../models/{model_file}'
    try:
        with open(path) as f:
            m = json.load(f)
        print(f"\n{m['dataset_name'].upper()} ({m['dataset_region']}) — {m['rows_total']:,} rows")
        print(f"  AUC: {m['metrics']['auc']:.4f}")
        print(f"  KS:  {m['metrics']['ks']:.4f}")
        print(f"  Top feature: {m['shap_top_features'][0]['feature']}")
    except FileNotFoundError:
        print(f'  {model_file} not found — run training first')

---

**Next steps:**
- Read [FINDINGS.md](../FINDINGS.md) for the full research results
- See [docs/research_paper.md](../docs/research_paper.md) for methodology
- Run `python -m workflow.training quickstart --dataset uci --jurisdiction US` to train from scratch
- Check [examples/bring_your_own_data/](bring_your_own_data/) to plug in your bank's data